# Python 知识

## telegram 库
`telegram` 库是 `python-telegram-bot` 框架的核心模块，为开发 Telegram 机器人提供了丰富的 API 封装。使开发者可以用 Python 轻松实现与 Telegram 服务器的交互，包括收发消息、处理命令、管理群组、发送多媒体等功能。
```mermaid
flowchart TD
    A[用户在Telegram发送消息/命令] --> B[Telegram服务器]
    B --> C[你的机器人程序]
    C --> D[消息分发器 Dispatcher]
    D --> E1[命令处理器 CommandHandler]
    D --> E2[消息处理器 MessageHandler]
    D --> E3[其他处理器]
    E1 & E2 & E3 --> F[你的回调函数]
    F --> G[处理结果/回复消息]
    G --> C
    C --> B
    B --> A

# 代码分析

## 类继承关系
```mermaid
classDiagram
    class Handler
    class LogHandler
    class Configured
    class TelegramBot
    
    %% 继承关系
    Handler <|-- LogHandler
    Configured <|-- TelegramBot
```

## class LogHandler(Handler)`


### Handler
在 `python-telegram-bot` 库中，`Handler` 是所有消息处理器的基类。
- 消息筛选：Handler 决定自己是否要处理某条消息（Update），比如只处理文本消息、命令消息等。
- 消息处理：如果 Handler 决定处理该消息，会调用你绑定的回调函数，执行具体的业务逻辑。
- 统一管理：所有的 Handler 都可以被 Dispatcher 统一管理和调度，保证消息分发的灵活性和可扩展性。

### 源码
`check_update` 负责检查并处理传入的更新对象。这里用它来记录用户消息，但不实际处理消息。

参数：
- `update`：Telegram 更新对象，可能包含消息、回调查询等

返回：
- False：表示不处理此更新，让其他处理器继续处理
- None：表示此更新不适用于本处理器
- True/object：表示处理此更新（在日志处理器中不使用）

```python
class LogHandler(Handler):

    def check_update(self, update: object) -> tp.Optional[tp.Union[bool, object]]:
        if isinstance(update, Update) and update.effective_message:
            message = update.effective_message  # 获取有效消息对象
            message_type = effective_message_type(message)  # 获取消息类型
            
            # 如果消息类型有效，记录日志
            if message_type is not None:
                if message_type == 'text':
                    # 记录文本消息内容
                    logger.info(f"{message.chat_id} - User: \"%s\"", message.text)
                else:
                    # 记录非文本消息类型
                    logger.info(f"{message.chat_id} - User: %s", message_type)
            return False  # 不处理此更新，让其他处理器继续处理
        return None  # 此更新不适用于本处理器
```

## send_action

用于装饰函数（这里的函数是处理命令）：处理命令时向用户显示机器人的状态（如"正在输入..."、"正在上传照片..."等）

参数：
- `action`：要发送的动作类型，如：
  - `typing`：正在输入
  - `upload_photo`：正在上传照片
  - `record_video`：正在录制视频
  - `upload_video`：正在上传视频
  - `record_audio`：正在录制音频
  - `upload_audio`：正在上传音频
  - `upload_document`：正在上传文档
  - `find_location`：正在查找位置

返回：装饰后的回调方法

### 源码
```python
def send_action(action: str) -> tp.Callable:

    def decorator(func: tp.Callable) -> tp.Callable:
        @wraps(func)
        def command_func(self, update: Update, context: CallbackContext, *args, **kwargs) -> tp.Callable:
            # 如果存在有效聊天，发送聊天动作状态
            if update.effective_chat:
                context.bot.send_chat_action(chat_id=update.effective_chat.id, action=action)
            # 调用原始函数并返回结果
            return func(self, update, context, *args, **kwargs)

        return command_func

    return decorator
```

## self_decorator
将机器人实例 `self` 绑定到回调函数 `func`，绑定后的函数可以访问该实例。

### 源码
```python
def self_decorator(self, func: tp.Callable) -> tp.Callable:

    def command_func(update, context, *args, **kwargs):
        return func(self, update, context, *args, **kwargs)

    return command_func
```

## class TelegramBot(Configured)

### `__init__`
- 创建更新器：`Updater` 对象 `self._updater`
- 获取调度器：`self._dispatcher = self.updater.dispatcher`
- 注册各种处理器

#### 源码
```python
def __init__(self, giphy_kwargs: tp.KwargsLike = None, **kwargs) -> None:
    from vectorbt._settings import settings
    telegram_cfg = settings['messaging']['telegram']
    giphy_cfg = settings['messaging']['giphy']
    # 调用父类构造函数，初始化配置管理功能
    Configured.__init__(
        self,
        giphy_kwargs=giphy_kwargs,
        **kwargs
    )

    # Resolve kwargs
    giphy_kwargs = merge_dicts(giphy_cfg, giphy_kwargs)
    self.giphy_kwargs = giphy_kwargs
    default_kwargs = dict()
    passed_kwargs = dict()
    for k in get_func_kwargs(Updater.__init__):
        if k in telegram_cfg:
            default_kwargs[k] = telegram_cfg[k]
        if k in kwargs:
            passed_kwargs[k] = kwargs.pop(k)
    # 合并参数，用户参数优先
    updater_kwargs = merge_dicts(default_kwargs, passed_kwargs)
    # 处理数据持久化配置
    persistence = updater_kwargs.pop('persistence', None)
    if isinstance(persistence, str):
        persistence = PicklePersistence(persistence)
    # 处理默认配置
    defaults = updater_kwargs.pop('defaults', None)
    if isinstance(defaults, dict):
        defaults = Defaults(**defaults)

    # 创建Updater对象（持久化的更新器）
    logger.info("Initializing bot")
    self._updater = Updater(persistence=persistence, defaults=defaults, **updater_kwargs)

    # 获取调度器以注册处理器
    self._dispatcher = self.updater.dispatcher

    # 注册各种处理器（按优先级顺序）
    self.dispatcher.add_handler(self.log_handler)
    self.dispatcher.add_handler(CommandHandler('start', self.start_callback))
    self.dispatcher.add_handler(CommandHandler("help", self.help_callback))
    # 注册自定义处理器
    for handler in self.custom_handlers:
        self.dispatcher.add_handler(handler)
    # 注册系统处理器
    self.dispatcher.add_handler(MessageHandler(Filters.status_update.migrate, self.chat_migration_callback))
    self.dispatcher.add_handler(MessageHandler(Filters.command, self.unknown_callback))
    self.dispatcher.add_error_handler(self_decorator(self, self.__class__.error_callback))

    # 初始化机器人数据
    if 'chat_ids' not in self.dispatcher.bot_data:
        self.dispatcher.bot_data['chat_ids'] = []
    else:
        logger.info("Loaded chat ids %s", str(self.dispatcher.bot_data['chat_ids']))
```

### log_handler
获取日志处理器实例。
```python
@property
def log_handler(self) -> LogHandler:
    return LogHandler(lambda update, context: None)
```

### custom_handlers
获取自定义处理器列表。
```python
@property
def custom_handlers(self) -> tp.Iterable[Handler]:

    return ()
```